[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ScriptsRemote/Transcend_SummerSchool/blob/main/Day3_Remote_Sensing_Fundamentals_and_Indices/02_water_quality_proxies.ipynb)

# Day 3 - Fundamentals of Remote Sensing & Spectral Indices
## Session 2: Water Quality Proxies

| | |
|---|---|
| **Fecha / Date** | 2026-10-14 (Wednesday / Miercoles) |
| **Horario / Time** | 10:40 - 12:10 |
| **Entidad / Entity** | UoM |
| **Instructores / Instructors** | Sebastian, Crhistian |

### Contenido / Content

> Water-Quality Proxies from Satellite Imagery

### Objetivos de aprendizaje / Learning objectives

- [ ] Explain why a water-quality index is a **proxy** and not a measurement, and what would
      be needed to turn one into a concentration
- [ ] Build a water mask with MNDWI, choose its threshold from a histogram instead of by
      habit, and **erode the shoreline** to remove mixed pixels
- [ ] Compute NDTI and NDCI and apply them **only inside the water body**, and show what
      the same indices produce when that rule is ignored
- [ ] Extract monthly NDTI and NDCI series for two contrasting sites and read the difference
      between them
- [ ] List the sources of error that affect these proxies over a high-altitude lake

### Datos necesarios / Required data

No local files. Imagery is read from the Earth Engine catalogue:
`COPERNICUS/S2_SR_HARMONIZED` (Sentinel-2 Level-2A surface reflectance, 2017 to present).

The Earth Engine account from Day 2, session 4 is required. This session continues in the
same study area as session 1 of today.

---

## 2.1 - Teoria / Theory: what a water-quality proxy is

### The optical problem

Over land, most of what the sensor receives comes from the surface. Over water it does not.
Water absorbs strongly across the near and short-wave infrared, so a clear lake reflects
only a few percent of the incoming light. The signal that actually carries information about
what is *in* the water - algae, sediment, dissolved matter - is a small part of an already
small number, and it sits underneath the light scattered by the atmosphere, the light
reflected off the surface, and in shallow water the light returning from the bottom.

This has a practical consequence you should carry through the whole session: **everything
we compute here is fragile**. The indices work, they are useful, and they are routinely used
in operational monitoring - but they are much more sensitive to processing errors than NDVI
ever is over a crop field.

### Three indices, three jobs

| Index | Formula (Sentinel-2) | Responds to | Source |
|---|---|---|---|
| **MNDWI** | $(B3 - B11)/(B3 + B11)$ | open water vs everything else | Xu (2006) |
| **NDTI** | $(B4 - B3)/(B4 + B3)$ | turbidity, suspended sediment | Lacaux et al. (2007) |
| **NDCI** | $(B5 - B4)/(B5 + B4)$ | chlorophyll-a, algal biomass | Mishra & Mishra (2012) |

MNDWI is **not** a water-quality index. Its job in this notebook is to answer a prior
question - *which pixels are water?* - so that the other two are applied where they mean
something. That ordering is the methodological core of the session.

---

### NDTI - turbidity

$$\text{NDTI} = \frac{\rho_{RED} - \rho_{GREEN}}{\rho_{RED} + \rho_{GREEN}}
= \frac{B4 - B3}{B4 + B3}$$

Clear water reflects more in the green than in the red, so NDTI is negative. As suspended
mineral particles increase, they scatter light back across the visible spectrum, but
proportionally more in the red - so the red rises faster than the green and NDTI increases,
crossing towards and past zero in very turbid water.

*What it confuses.* Anything else that raises red reflectance: a shallow sandy bottom, a
bloom of brown-pigmented algae, sun glint on a rippled surface. NDTI does not distinguish
mineral turbidity from organic turbidity.

---

### NDCI - chlorophyll-a

$$\text{NDCI} = \frac{\rho_{RED\,EDGE} - \rho_{RED}}{\rho_{RED\,EDGE} + \rho_{RED}}
= \frac{B5 - B4}{B5 + B4}$$

This one depends on a specific piece of physics. Chlorophyll-a absorbs strongly near
**665 nm** (Sentinel-2 B4, red). Just beyond it, near **705 nm** (B5, the first red-edge
band), absorption collapses and cell scattering produces a reflectance peak. The height of
that peak relative to the red trough grows with chlorophyll concentration. NDCI measures
exactly that contrast.

Two things follow:

- **The red edge is what makes this possible.** Landsat has no band at 705 nm, which is why
  NDCI is a Sentinel-2 index. Session 1 could use Landsat because water extent only needs
  green, NIR and SWIR; chlorophyll needs the red edge.
- **B5 is a 20 m band.** B4 and B3 are 10 m. Earth Engine resamples silently when you combine
  them, so the effective resolution of NDCI is 20 m, not 10 m. We set `scale = 20` everywhere
  to keep this explicit.

*What it confuses.* Any vegetation. Floating macrophytes - and Lago Menor has extensive
*totorales* and duckweed - have a far stronger red edge than any phytoplankton bloom, so they
produce very high NDCI while not being phytoplankton at all. Emergent vegetation near the
shore is the single most common way to read a bloom that is not there.

---

### Proxy, not measurement

An index is a number between -1 and 1. Chlorophyll-a is a concentration in mg/m3. Turbidity
is measured in NTU. Nothing in the arithmetic above converts one into the other.

Turning a proxy into a measurement requires **in-situ samples taken at the same time as the
overpass**, a regression fitted on those pairs, and an independent set held back to test it.
That regression is valid for the water body it was fitted on, and often only for the season
it was fitted in, because the relationship depends on the local mix of algae, sediment and
dissolved organic matter.

What the indices *are* good for, without any calibration:

- **Spatial pattern** - where in the lake the values are highest, today.
- **Change over time** - whether a site is getting worse or better relative to itself.
- **Comparison between sites** - one location against another, in the same image.

Use them for those three questions and they are informative. Report them as a concentration
without calibration and they are not defensible.

---

### The atmosphere: the caveat that matters most

We use Sentinel-2 **Level-2A** (`S2_SR_HARMONIZED`), which is surface reflectance corrected
by Sen2Cor. Sen2Cor was designed and validated **for land surfaces**. Over dark water the
correction is being asked to remove an atmospheric contribution that is comparable to, or
larger than, the surface signal itself, and its aerosol retrieval degrades. It is not unusual
to find slightly negative reflectances over clear deep water in L2A.

Operational water-quality work uses corrections built for water - ACOLITE, C2RCC, POLYMER -
which are not available in the Earth Engine catalogue. So:

> Within one image, the relative pattern is informative. Between images months apart, part of
> any difference you see is atmospheric correction, not water. This is the main reason we
> compare **two sites in the same image** rather than trusting absolute values.

### Why this matters here: Cohana Bay and the Katari

Lake Titicaca is naturally oligotrophic, but the southern shallow basin is not uniformly so.
The **Katari** river drains El Alto and Viacha and discharges largely untreated wastewater
into **Cohana Bay**, on the south-eastern shore of Lago Menor. Nutrient enrichment there has
been documented for over two decades, with proliferation of duckweed (*Lemna gibba*) and an
altered algal community.

In **March-April 2015** the basin experienced its first recorded large algal bloom, dominated
by *Carteria* sp., killing an estimated two tonnes of fish, amphibians and birds. It followed
exceptionally intense rainfall - April 2015 precipitation was 35% to 540% above other years
in 2013-2016 - which flushed nutrients from the watershed.

Note the date. The Sentinel-2 Level-2A archive begins in **2017**, so that specific event is
not in the data we are about to use. That is not a footnote - it is the argument for routine
monitoring: the instrument that would have documented the event was not yet recording it.

---

## 2.2 - Setup and study area

Same area as session 1. The maps are centred on **Lago Menor (Wi&ntilde;aymarka)**, the shallow
southern basin, because that is where both turbidity and chlorophyll actually vary.

In [ ]:
# ---------------------------------------------------------------------------
# Libraries (uncomment the pip line on a fresh runtime, e.g. Google Colab)
# ---------------------------------------------------------------------------
#%pip install -q earthengine-api geemap

import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt

print("ee", ee.__version__, "| geemap", geemap.__version__)

In [ ]:
# ---------------------------------------------------------------------------
# Authenticate and initialise Earth Engine
# ---------------------------------------------------------------------------
# >>> REPLACE with your own Cloud project id <<<
EE_PROJECT = "ee-christhianscunha"

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

print("Earth Engine connected - project:", EE_PROJECT)

In [ ]:
# ---------------------------------------------------------------------------
# Study area - same rectangle as session 1, plus the Lago Menor display window
# ---------------------------------------------------------------------------
ROI = ee.Geometry.Rectangle(
    [-70.8017, -16.8074, -68.1623, -15.0219],   # west, south, east, north
    None, False
)

# Lago Menor (Winaymarka): the shallow southern basin, where we work
LAGO_MENOR = ee.Geometry.Rectangle([-69.15, -16.65, -68.55, -16.10], None, False)

CENTER_LAT, CENTER_LON = -16.35, -68.90

print("ROI        : {:,.0f} km2".format(ROI.area(maxError=1000).divide(1e6).getInfo()))
print("Lago Menor : {:,.0f} km2".format(LAGO_MENOR.area(maxError=100).divide(1e6).getInfo()))

### The Sentinel-2 collection

Five bands are needed: **B3** (green, 10 m), **B4** (red, 10 m), **B5** (red edge 1, 20 m)
and **B11** (SWIR1, 20 m) for the indices, plus **B2** (blue) so that a true-colour image is
available to check every result against.

The cloud filter is tighter than in the vegetation notebooks - 40% instead of 60 or 80.
Over land a few clouds degrade a composite; over water they destroy it, because a thin cloud
is far brighter than the water underneath and the median of a few contaminated observations
is no longer the water.

Two classes are deliberately **kept** by the per-pixel mask: SCL 6 (water) and SCL 5 (bare
soil), the latter because shallow turbid water is frequently classified as bare soil by
Sen2Cor. Discarding it would systematically remove the most turbid pixels - exactly the ones
we are trying to measure.

In [ ]:
# ---------------------------------------------------------------------------
# Sentinel-2 Level-2A: filter, mask clouds, rescale to reflectance
# ---------------------------------------------------------------------------
import datetime

START = "2018-01-01"
END = datetime.date.today().replace(day=1).isoformat()   # first day of this month
CLOUD_MAX = 40

BANDS = ["B2", "B3", "B4", "B5", "B11"]   # blue, green, red, red edge 1, SWIR1

def mask_s2(image):
    # SCL classes to discard: 1 saturated, 3 shadow, 8-9 cloud, 10 cirrus, 11 snow.
    # SCL 5 (bare soil) and 6 (water) are kept on purpose - see the note above.
    scl = image.select("SCL")
    clear = (scl.neq(1).And(scl.neq(3)).And(scl.neq(8))
                .And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11)))

    optical = image.select(BANDS).multiply(0.0001).updateMask(clear)

    # multiply() drops the metadata, so the date has to be copied back
    return ee.Image(optical.copyProperties(image, ["system:time_start"]))

s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(LAGO_MENOR)
        .filterDate(START, END)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", CLOUD_MAX))
        .map(mask_s2))

print("Period       :", START, "to", END)
print("Cloud filter : <", CLOUD_MAX, "%")
print("Images       :", s2.size().getInfo())

In [ ]:
# ---------------------------------------------------------------------------
# Reference image: the least cloudy scene, used for all the maps in 2.3 and 2.4
# ---------------------------------------------------------------------------
reference = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
               .filterBounds(LAGO_MENOR)
               .filterDate(START, END)
               .sort("CLOUDY_PIXEL_PERCENTAGE")
               .first())

image = ee.Image(mask_s2(reference)).clip(LAGO_MENOR)

print("Date        :", ee.Date(image.get("system:time_start")).format("YYYY-MM-dd").getInfo())
print("Cloud cover :", round(reference.get("CLOUDY_PIXEL_PERCENTAGE").getInfo(), 1), "%")

---

## 2.3 - The water mask, and why the shoreline has to go

### Step 1 - MNDWI

$$\text{MNDWI} = \frac{B3 - B11}{B3 + B11}$$

Water absorbs almost completely in the SWIR, so B11 is near zero over water and MNDWI goes
strongly positive. Over land, vegetation and soil both reflect in the SWIR, so MNDWI is
negative. The separation is large, which is what makes a single threshold workable.

MNDWI is preferred over NDWI here because it replaces the near infrared with the SWIR, which
removes the confusion between water and built-up surfaces - relevant on a shore with
settlements.

In [ ]:
# ---------------------------------------------------------------------------
# MNDWI = (B3 - B11) / (B3 + B11)
# ---------------------------------------------------------------------------
mndwi = image.normalizedDifference(["B3", "B11"]).rename("MNDWI")

vis_mndwi = {"min": -0.6, "max": 0.6,
             "palette": ["#8c5a2b", "#d9c9a3", "#f7f7f7", "#7fb0d0", "#0a3d6b"]}

Map1 = geemap.Map(center=[CENTER_LAT, CENTER_LON], zoom=10)
Map1.add_basemap("SATELLITE")
Map1.addLayer(image, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 0.25}, "True colour")
Map1.addLayer(mndwi, vis_mndwi, "MNDWI")
Map1.add_colorbar(vis_mndwi, label="MNDWI")

Map1

### Step 2 - Choose the threshold, do not inherit it

`MNDWI > 0` is the conventional threshold and it is usually reasonable, but it is a
convention, not a law. The honest way to set it is to look at the distribution: over a scene
containing a large lake, MNDWI is **bimodal** - a land mode well below zero and a water mode
well above it - and the threshold belongs in the valley between them.

If your histogram has no valley, the threshold is arbitrary and you should say so in your
results.

In [ ]:
# ---------------------------------------------------------------------------
# Histogram of MNDWI over Lago Menor: find the valley between land and water
# ---------------------------------------------------------------------------
hist = mndwi.reduceRegion(
    reducer=ee.Reducer.fixedHistogram(-1, 1, 100),
    geometry=LAGO_MENOR, scale=60, maxPixels=1e9).getInfo()["MNDWI"]

edges = [h[0] for h in hist]
counts = [h[1] for h in hist]

INK, MUTED, GRID = "#0b0b0b", "#52514e", "#d9d8d2"
BLUE, ORANGE, RED = "#1f6aa5", "#e07b39", "#c0392b"

MNDWI_THRESHOLD = 0.0        # <<< change this after looking at the histogram

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(edges, counts, width=0.02, color=BLUE, zorder=3)
ax.axvline(MNDWI_THRESHOLD, color=RED, linewidth=1.5, linestyle="--", zorder=4,
           label="threshold = {:.2f}".format(MNDWI_THRESHOLD))
ax.set_xlabel("MNDWI", fontsize=10, color=MUTED)
ax.set_ylabel("Pixel count", fontsize=10, color=MUTED)
ax.set_title("MNDWI distribution over Lago Menor", fontsize=11, color=INK, loc="left", pad=6)
ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color(GRID)
ax.spines["bottom"].set_color(GRID)
ax.tick_params(colors=MUTED, labelsize=9)
ax.legend(frameon=False, fontsize=9, labelcolor=INK)

fig.subplots_adjust(left=0.09, right=0.98, top=0.90, bottom=0.14)
plt.show()

### Step 3 - Erode the shoreline

This is the step that separates a defensible water-quality map from a decorative one.

A 20 m pixel on the shoreline contains water **and** land. Its reflectance is a mixture, and
the mixture is dominated by the land, because land is much brighter. A shoreline pixel can
pass an MNDWI threshold and still be half beach, half *totora* - and when NDCI is applied to
it, the vegetation's red edge produces a large positive value that reads as an algal bloom.

There is a second, subtler effect. Even a pixel that is **entirely** water, but close to
land, is contaminated: photons reflected from the bright land surface scatter in the
atmosphere into the sensor's line of sight for the neighbouring water pixel. This is the
**adjacency effect**, and it extends tens to hundreds of metres offshore.

The fix is simple: shrink the mask inwards. `focal_min` replaces each pixel by the minimum
of its neighbourhood, so any pixel within `EROSION_M` metres of a non-water pixel becomes
non-water. We lose a strip of real water along the whole shore, and we accept that loss,
because those are the pixels we can least trust.

In [ ]:
# ---------------------------------------------------------------------------
# Water mask: threshold, then erode inwards by EROSION_M metres
# ---------------------------------------------------------------------------
EROSION_M = 60          # width of the shoreline strip removed, in metres

water_raw = mndwi.gt(MNDWI_THRESHOLD).rename("water")

# focal_min: a pixel survives only if every pixel within EROSION_M metres is also water
water_clean = water_raw.focal_min(radius=EROSION_M, units="meters").rename("water")

# How much water did the erosion cost us?
pixel_area = ee.Image.pixelArea()
areas = (water_raw.multiply(pixel_area).rename("raw")
         .addBands(water_clean.multiply(pixel_area).rename("clean"))
         .reduceRegion(reducer=ee.Reducer.sum(), geometry=LAGO_MENOR,
                       scale=20, maxPixels=1e10)).getInfo()

raw_km2 = areas["raw"] / 1e6
clean_km2 = areas["clean"] / 1e6

print("Water area, raw mask    : {:8,.1f} km2".format(raw_km2))
print("Water area, eroded mask : {:8,.1f} km2".format(clean_km2))
print("Removed along the shore : {:8,.1f} km2  ({:.1f}%)".format(
    raw_km2 - clean_km2, 100 * (raw_km2 - clean_km2) / raw_km2))

In [ ]:
# ---------------------------------------------------------------------------
# The two masks side by side - zoom into a bay to see the difference
# ---------------------------------------------------------------------------
vis_raw = {"palette": ["#e07b39"]}
vis_clean = {"palette": ["#1f6aa5"]}

Map2 = geemap.Map(center=[CENTER_LAT, CENTER_LON], zoom=11)
Map2.add_basemap("SATELLITE")
Map2.addLayer(water_raw.selfMask(), vis_raw, "MNDWI mask (raw)")
Map2.addLayer(water_clean.selfMask(), vis_clean, "MNDWI mask (eroded)")

# Orange visible around the edge = the strip that was discarded
Map2.add_legend(title="Water mask",
                legend_dict={"discarded shoreline": "#e07b39", "kept": "#1f6aa5"})

Map2

---

## 2.4 - NDTI and NDCI, inside the water only

Now the indices. Two lines of arithmetic, and then the line that matters:
`.updateMask(water_clean)`.

In [ ]:
# ---------------------------------------------------------------------------
# NDTI = (B4 - B3) / (B4 + B3)      turbidity
# NDCI = (B5 - B4) / (B5 + B4)      chlorophyll-a
# ---------------------------------------------------------------------------
ndti = image.normalizedDifference(["B4", "B3"]).rename("NDTI")
ndci = image.normalizedDifference(["B5", "B4"]).rename("NDCI")

# The whole point of the session: the indices only exist inside the water body
ndti_water = ndti.updateMask(water_clean)
ndci_water = ndci.updateMask(water_clean)

vis_ndti = {"min": -0.4, "max": 0.1,
            "palette": ["#0a3d6b", "#7fb0d0", "#f0e2c0", "#d6a04a", "#8c4a1e"]}
vis_ndci = {"min": -0.2, "max": 0.3,
            "palette": ["#2c5aa0", "#8fc4e0", "#f7f3d8", "#94c355", "#1a6b2a"]}

print("NDTI: blue = clear, brown = turbid")
print("NDCI: blue = low chlorophyll, green = high chlorophyll")

### What happens when you skip the mask

Before looking at the result, look at the mistake. The two panels below show **NDCI computed
over everything**, and **NDCI computed inside the water mask**.

Unmasked, the land is bright green: terrestrial vegetation has a red edge an order of
magnitude stronger than any phytoplankton. If you were handed the left panel without the
mask, every field around the lake would read as a severe algal bloom.

This is why NDTI and NDCI are applied to the water body and to nothing else. The index has
no meaning outside the medium it was designed for.

In [ ]:
# ---------------------------------------------------------------------------
# The same index, with and without the water mask
# ---------------------------------------------------------------------------
geemap.linked_maps(
    rows=1, cols=2, height="450px",
    center=[CENTER_LAT, CENTER_LON], zoom=11,
    ee_objects=[ndci, ndci_water],
    vis_params=[vis_ndci, vis_ndci],
    labels=["NDCI - no mask (WRONG)", "NDCI - inside the water mask"],
    label_position="topright",
)

In [ ]:
# ---------------------------------------------------------------------------
# NDTI and NDCI over the lake, both masked to water
# ---------------------------------------------------------------------------
Map3 = geemap.Map(center=[CENTER_LAT, CENTER_LON], zoom=11)
Map3.add_basemap("SATELLITE")
Map3.addLayer(image, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 0.25}, "True colour", False)
Map3.addLayer(ndti_water, vis_ndti, "NDTI (turbidity)")
Map3.addLayer(ndci_water, vis_ndci, "NDCI (chlorophyll-a)", False)
Map3.add_colorbar(vis_ndti, label="NDTI - clear (blue) to turbid (brown)")

Map3

### Reading the two maps

Turn the layers on and off and work through this:

1. **River mouths.** Where a river enters, expect a plume of high NDTI fading offshore. The
   south-eastern shore of Lago Menor, where the **Katari** discharges, is the one to check.
2. **NDTI against NDCI.** Where both are high, the water is turbid *and* productive - typical
   of nutrient-rich inflow. Where NDTI is high and NDCI is not, the turbidity is mineral.
   Where NDCI is high and NDTI is not, suspect algae - or vegetation the mask failed to remove.
3. **The shoreline, again.** Look at what is left immediately inside the eroded mask. If a
   thin rim of extreme values survives, the erosion was not wide enough: raise `EROSION_M`
   to 100 or 150 and re-run.
4. **The distribution, not just the colours.** The histogram below shows how the values are
   actually spread. A long right tail in NDCI means a small area with very high values -
   which is what a localised bloom looks like, and what a patch of missed vegetation looks
   like too.

In [ ]:
# ---------------------------------------------------------------------------
# Distribution of the two indices inside the water mask
# ---------------------------------------------------------------------------
def index_histogram(index_image, band):
    h = index_image.reduceRegion(
        reducer=ee.Reducer.fixedHistogram(-0.6, 0.6, 80),
        geometry=LAGO_MENOR, scale=60, maxPixels=1e9).getInfo()[band]
    return [x[0] for x in h], [x[1] for x in h]

edges_ndti, counts_ndti = index_histogram(ndti_water, "NDTI")
edges_ndci, counts_ndci = index_histogram(ndci_water, "NDCI")

fig, (axa, axb) = plt.subplots(1, 2, figsize=(12, 4))

for ax, e, c, name, colour in [(axa, edges_ndti, counts_ndti, "NDTI", "#8c4a1e"),
                               (axb, edges_ndci, counts_ndci, "NDCI", "#1a6b2a")]:
    ax.bar(e, c, width=0.015, color=colour, zorder=3)
    ax.axvline(0, color=MUTED, linewidth=1, linestyle="--", zorder=4)
    ax.set_xlabel(name, fontsize=10, color=MUTED)
    ax.set_title(name + " inside the water mask", fontsize=11, color=INK, loc="left", pad=6)
    ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(GRID)
    ax.spines["bottom"].set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=9)

axa.set_ylabel("Pixel count", fontsize=10, color=MUTED)
fig.subplots_adjust(left=0.08, right=0.98, top=0.88, bottom=0.14, wspace=0.18)
plt.show()

---

## 2.5 - Application: two sites, followed month by month

Absolute index values drift with the atmospheric correction, so the defensible design is a
**comparison between two sites measured in the same image**: whatever the atmosphere did
that day, it did to both.

- **Site A - impacted.** Near the south-eastern shore of Lago Menor, in the area influenced
  by the Katari discharge (Cohana).
- **Site B - reference.** Open water in the middle of Lago Menor, away from any inflow.

The default boxes below are **approximate**. Draw your own on the map - place Site A on a
patch of high NDCI or NDTI that you identified in section 2.4 - and use the printed water-pixel
count to confirm that whatever box you end up with actually falls on water.

In [ ]:
# ---------------------------------------------------------------------------
# Draw Site A here (the impacted site). Leave empty to use the default box.
# ---------------------------------------------------------------------------
Map_draw = geemap.Map(center=[-16.40, -68.70], zoom=11)
Map_draw.add_basemap("SATELLITE")
Map_draw.addLayer(ndci_water, vis_ndci, "NDCI (chlorophyll-a)")
Map_draw.addLayer(ndti_water, vis_ndti, "NDTI (turbidity)", False)

Map_draw

In [ ]:
# ---------------------------------------------------------------------------
# The two sites, and a check that both of them are on water
# ---------------------------------------------------------------------------
# Approximate boxes - verify them on the map and move them if needed
SITE_A_DEFAULT = ee.Geometry.Rectangle([-68.68, -16.44, -68.58, -16.36], None, False)
SITE_B = ee.Geometry.Rectangle([-68.95, -16.32, -68.86, -16.24], None, False)

SITE_A = Map_draw.user_roi if Map_draw.user_roi is not None else SITE_A_DEFAULT

def water_area_in(site):
    m2 = water_clean.multiply(pixel_area).reduceRegion(
        reducer=ee.Reducer.sum(), geometry=site, scale=20, maxPixels=1e9).getInfo()["water"]
    return m2 / 1e6

for name, site in [("Site A (impacted)", SITE_A), ("Site B (reference)", SITE_B)]:
    box_km2 = site.area(maxError=10).divide(1e6).getInfo()
    wet_km2 = water_area_in(site)
    flag = "OK" if wet_km2 > 0.3 * box_km2 else "<-- CHECK: mostly land, move this box"
    print("{:20s} box {:6.1f} km2 | water {:6.1f} km2  {}".format(name, box_km2, wet_km2, flag))

### Monthly composites

One median composite per month, exactly as in session 1. Two things are done differently
because this is water:

- The **water mask is recomputed for every month**, from that month's own MNDWI. The lake
  level moves, and a mask frozen from one date would keep counting pixels that are now beach.
- The indices are masked **before** any statistic is taken, so the monthly mean is a mean
  over water pixels only.

In [ ]:
# ---------------------------------------------------------------------------
# One composite per month: MNDWI mask (recomputed), then NDTI and NDCI
# ---------------------------------------------------------------------------
n_months = ee.Date(END).difference(ee.Date(START), "month").round()
offsets = ee.List.sequence(0, n_months.subtract(1))

def monthly_indices(offset):
    start = ee.Date(START).advance(offset, "month")
    month_images = s2.filterDate(start, start.advance(1, "month"))
    composite = month_images.median()

    m = composite.normalizedDifference(["B3", "B11"])
    water = m.gt(MNDWI_THRESHOLD).focal_min(radius=EROSION_M, units="meters")

    t = composite.normalizedDifference(["B4", "B3"]).rename("NDTI")
    c = composite.normalizedDifference(["B5", "B4"]).rename("NDCI")

    return (t.addBands(c).updateMask(water)
             .set("date", start.format("YYYY-MM-dd"))
             .set("n_images", month_images.size()))

# Months with no acquisition are dropped before anything is computed on them
monthly = (ee.ImageCollection(offsets.map(monthly_indices))
             .filter(ee.Filter.gt("n_images", 0)))

print("Months requested:", n_months.getInfo())
print("Months with data:", monthly.size().getInfo())

In [ ]:
# ---------------------------------------------------------------------------
# Mean NDTI and NDCI inside the water mask, for one site, month by month
# ---------------------------------------------------------------------------
def series_for(site, label):
    def month_stats(image):
        # mean and count in one pass: count tells us how many water pixels
        # were actually observed, which is the quality check
        stats = image.reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True),
            geometry=site, scale=20, maxPixels=1e9)

        return ee.Feature(None, {
            "date": image.get("date"),
            "ndti": stats.get("NDTI_mean"),
            "ndci": stats.get("NDCI_mean"),
            "water_px": stats.get("NDTI_count"),
        })

    rows = [f["properties"] for f in monthly.map(month_stats).getInfo()["features"]]

    out = pd.DataFrame(rows)
    out["date"] = pd.to_datetime(out["date"])
    out = out.dropna().sort_values("date").set_index("date")
    out["site"] = label
    return out

# Each call takes 1-3 minutes
df_a = series_for(SITE_A, "A - impacted")
df_b = series_for(SITE_B, "B - reference")

print("Site A: {} months".format(len(df_a)))
print("Site B: {} months".format(len(df_b)))
df_a.head()

### Drop the months with too few water pixels

A month where cloud left only a handful of clear water pixels in the box produces a mean
computed on almost nothing. `water_px` is what catches that: we keep only months where at
least half of the site's typical water pixel count was observed.

The rule is deliberately expressed relative to each site's own median, because the two boxes
are different sizes.

In [ ]:
# ---------------------------------------------------------------------------
# Quality filter: keep months with enough observed water pixels
# ---------------------------------------------------------------------------
MIN_FRACTION = 0.50

def drop_poor_months(frame):
    typical = frame["water_px"].median()
    keep = frame[frame["water_px"] >= MIN_FRACTION * typical].copy()
    print("{:16s} {:3d} months -> {:3d} kept (median {:,.0f} water pixels)".format(
        frame["site"].iloc[0], len(frame), len(keep), typical))
    return keep

good_a = drop_poor_months(df_a)
good_b = drop_poor_months(df_b)

In [ ]:
# ---------------------------------------------------------------------------
# The two sites, both indices
# ---------------------------------------------------------------------------
fig, (ax_t, ax_c) = plt.subplots(2, 1, figsize=(12, 7.5), sharex=True,
                                 gridspec_kw={"hspace": 0.22})

for ax, column, title, unit in [
        (ax_t, "ndti", "NDTI - turbidity proxy", "clear (low) to turbid (high)"),
        (ax_c, "ndci", "NDCI - chlorophyll-a proxy", "low to high algal biomass")]:

    ax.plot(good_a.index, good_a[column], color=ORANGE, linewidth=1.8,
            marker="o", markersize=3.5, markeredgecolor="white", markeredgewidth=0.6,
            label="Site A - impacted", zorder=3)
    ax.plot(good_b.index, good_b[column], color=BLUE, linewidth=1.8,
            marker="o", markersize=3.5, markeredgecolor="white", markeredgewidth=0.6,
            label="Site B - reference", zorder=3)

    ax.set_title(title, fontsize=11, color=INK, loc="left", pad=6)
    ax.set_ylabel(unit, fontsize=9, color=MUTED)
    ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color(GRID)
    ax.spines["bottom"].set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.legend(frameon=False, fontsize=9, ncol=2, loc="upper left", labelcolor=INK)

fig.suptitle("Water-quality proxies, Lago Menor", fontsize=13, color=INK,
             x=0.08, ha="left", y=0.97)
fig.subplots_adjust(left=0.08, right=0.98, top=0.90, bottom=0.07)
plt.show()

### The difference between the sites

The two curves above share whatever the atmosphere and the seasons did to both. Subtracting
one from the other removes most of that shared component and leaves the part that is specific
to Site A.

A difference that stays consistently above zero is the signature of a locally impacted site.
A difference that only appears in certain months points at a seasonal driver - the rainy
season flushing the catchment, for instance, which is what preceded the 2015 bloom.

In [ ]:
# ---------------------------------------------------------------------------
# Site A minus Site B, month by month
# ---------------------------------------------------------------------------
diff = pd.DataFrame({
    "ndti": good_a["ndti"] - good_b["ndti"],
    "ndci": good_a["ndci"] - good_b["ndci"],
}).dropna()

fig, ax = plt.subplots(figsize=(12, 4.5))

ax.axhline(0, color=MUTED, linewidth=1, zorder=2)
ax.fill_between(diff.index, 0, diff["ndci"], color="#1a6b2a", alpha=0.25, zorder=3)
ax.plot(diff.index, diff["ndci"], color="#1a6b2a", linewidth=1.8, label="NDCI difference", zorder=4)
ax.plot(diff.index, diff["ndti"], color="#8c4a1e", linewidth=1.8, label="NDTI difference", zorder=4)

ax.set_ylabel("Site A minus Site B", fontsize=10, color=MUTED)
ax.set_title("How much worse is the impacted site than the reference?",
             fontsize=11, color=INK, loc="left", pad=6)
ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color(GRID)
ax.spines["bottom"].set_color(GRID)
ax.tick_params(colors=MUTED, labelsize=9)
ax.legend(frameon=False, fontsize=9, ncol=2, loc="upper left", labelcolor=INK)

fig.subplots_adjust(left=0.08, right=0.98, top=0.90, bottom=0.12)
plt.show()

print("Months compared        :", len(diff))
print("Mean NDCI difference   : {:+.4f}".format(diff["ndci"].mean()))
print("Mean NDTI difference   : {:+.4f}".format(diff["ndti"].mean()))
print("Months A above B (NDCI): {} of {}".format(int((diff['ndci'] > 0).sum()), len(diff)))

In [ ]:
# ---------------------------------------------------------------------------
# Average seasonal cycle of the difference
# ---------------------------------------------------------------------------
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
          "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

cycle = diff.groupby(diff.index.month).mean()

fig, ax = plt.subplots(figsize=(10, 4))

x = list(cycle.index)
ax.bar([i - 0.2 for i in x], cycle["ndti"], width=0.4, color="#8c4a1e", label="NDTI", zorder=3)
ax.bar([i + 0.2 for i in x], cycle["ndci"], width=0.4, color="#1a6b2a", label="NDCI", zorder=3)
ax.axhline(0, color=MUTED, linewidth=1, zorder=4)

ax.set_xticks(range(1, 13))
ax.set_xticklabels(MONTHS)
ax.set_ylabel("Mean difference (A - B)", fontsize=10, color=MUTED)
ax.set_title("In which months is the impacted site worst?", fontsize=11, color=INK,
             loc="left", pad=6)
ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color(GRID)
ax.spines["bottom"].set_color(GRID)
ax.tick_params(colors=MUTED, labelsize=9)
ax.legend(frameon=False, fontsize=9, ncol=2, labelcolor=INK)

fig.subplots_adjust(left=0.09, right=0.98, top=0.90, bottom=0.12)
plt.show()

print("Rainy season in the altiplano: roughly December to March.")

---

## 2.6 - From index to concentration, and why you should not

Published papers do give equations converting NDCI into chlorophyll-a. Mishra & Mishra (2012)
fitted a quadratic on data from turbid productive estuaries:

$$\text{Chl-}a\ [\text{mg m}^{-3}] = 14.039 + 86.115 \cdot \text{NDCI}
+ 194.325 \cdot \text{NDCI}^2$$

The cell below applies it. Run it, look at the numbers, and then read why they should not
leave this notebook.

> **Check the coefficients against the original paper before using them for anything.**
> They are reproduced here as a teaching example, not as a validated implementation.

In [ ]:
# ---------------------------------------------------------------------------
# Applying a published calibration - as a demonstration, not as a result
# ---------------------------------------------------------------------------
# Coefficients from Mishra & Mishra (2012), fitted on turbid productive
# ESTUARINE waters - not on a high-altitude oligotrophic lake.
A0, A1, A2 = 14.039, 86.115, 194.325

# Chl-a = A0 + A1 * NDCI + A2 * NDCI^2, written as plain image arithmetic
chla = (ndci_water.multiply(ndci_water).multiply(A2)   # A2 * NDCI^2
          .add(ndci_water.multiply(A1))                # + A1 * NDCI
          .add(A0)                                     # + A0
          .rename("chla"))

vis_chla = {"min": 0, "max": 60,
            "palette": ["#2c5aa0", "#8fc4e0", "#f7f3d8", "#f0a53c", "#b52d2d"]}

Map4 = geemap.Map(center=[CENTER_LAT, CENTER_LON], zoom=11)
Map4.add_basemap("SATELLITE")
Map4.addLayer(chla, vis_chla, "Chlorophyll-a (UNCALIBRATED)")
Map4.add_colorbar(vis_chla, label="Chl-a proxy [mg/m3] - NOT a measurement")

stats = chla.reduceRegion(
    reducer=ee.Reducer.percentile([5, 50, 95]),
    geometry=LAGO_MENOR, scale=60, maxPixels=1e9).getInfo()

print("Values produced by the equation over Lago Menor:")
print("  5th percentile : {:6.1f} mg/m3".format(stats["chla_p5"]))
print("  median         : {:6.1f} mg/m3".format(stats["chla_p50"]))
print("  95th percentile: {:6.1f} mg/m3".format(stats["chla_p95"]))

Map4

### Why those numbers are not chlorophyll-a

Four reasons, each sufficient on its own:

1. **The calibration is from somewhere else.** It was fitted on turbid estuarine waters whose
   optical composition - the mix of phytoplankton, mineral sediment and coloured dissolved
   organic matter - is nothing like a high-altitude oligotrophic lake at 3,810 m. A regression
   is only valid inside the range and the conditions it was fitted on.
2. **The quadratic has no floor.** At the low NDCI values typical of clear water the equation
   returns a positive number by construction, around 14 mg/m3, simply because that is its
   intercept. Clear Titicaca water is well below that. The model cannot represent the range we
   are in.
3. **The atmospheric correction is not water-grade.** Sen2Cor errors over dark water propagate
   straight into NDCI and are then amplified by the quadratic term.
4. **There is no validation.** No in-situ sample from this lake has been compared to any of
   these pixels.

**What you would need to do this properly.** Water samples collected within a few hours of a
Sentinel-2 overpass, spread across the range of conditions you want to measure and across
seasons; the matching index values extracted at those coordinates; a regression fitted on part
of the pairs and tested on the rest; and the reported error of that test published alongside
every map you produce. The Observatorio del Lago Titicaca (UMSA) and the field campaigns
around the Katari basin are the obvious sources of such data for this system.

Until then: **report the index, not the concentration.**

---

## 2.7 - Ejercicio / Exercise

**Consigna / Task**

Part 1 - *Test how much the mask decides the answer.*

Re-run sections 2.3 to 2.5 three times, changing one thing each time:

| Run | `MNDWI_THRESHOLD` | `EROSION_M` |
|---|---|---|
| 1 | 0.0 | 0 (no erosion) |
| 2 | 0.0 | 60 |
| 3 | 0.2 | 120 |

Record the mean NDCI of Site A for each run. Then answer: how large is the spread between the
three, compared with the difference between Site A and Site B that you measured in 2.5? If the
methodological choice moves the number as much as the phenomenon does, what can you still
legitimately conclude?

Part 2 - *Find and characterise a hotspot.*

1. Using the NDCI and NDTI maps of section 2.4, locate a patch of water with anomalously high
   values that is **not** at a river mouth.
2. Draw a polygon on it and extract its series.
3. Decide, with evidence, whether it is a bloom, floating vegetation, shallow bottom, or a mask
   failure. Useful checks: the true-colour image at the same date; whether the patch moves
   between months or stays put; whether NDTI and NDCI agree; how close it is to shore.

**Entregable / Deliverable**

A short report (2 pages) containing: the three-run table from Part 1, the time series and maps
for your hotspot, your conclusion on what it is, and an explicit paragraph listing which of
your statements would survive if the atmospheric correction were wrong by 0.01 in reflectance.

---

## Referencias / References

**Indices**

- Mishra, S., Mishra, D.R. (2012). Normalized difference chlorophyll index: A novel model for
  remote estimation of chlorophyll-a concentration in turbid productive waters. *Remote
  Sensing of Environment*, 117, 394-406. <https://doi.org/10.1016/j.rse.2011.10.016> (NDCI)
- Lacaux, J.P., Tourre, Y.M., Vignolles, C., Ndione, J.A., Lafaye, M. (2007). Classification
  of ponds from high-spatial resolution remote sensing: Application to Rift Valley Fever
  epidemics in Senegal. *Remote Sensing of Environment*, 106(1), 66-74. (NDTI)
- Xu, H. (2006). Modification of normalised difference water index (NDWI) to enhance open
  water features in remotely sensed imagery. *International Journal of Remote Sensing*,
  27(14), 3025-3033. (MNDWI)
- Gitelson, A. (1992). The peak near 700 nm on radiance spectra of algae and water:
  relationships of its magnitude and position with chlorophyll concentration.
  *International Journal of Remote Sensing*, 13(17), 3367-3373. (the red-edge peak NDCI uses)

**Lake Titicaca and the Katari basin**

- Chalar, G. et al. (2018). Algal bloom exacerbates hydrogen sulfide and methylmercury
  contamination in the emblematic high-altitude Lake Titicaca. *Geosciences*, 8(12), 438.
  <https://doi.org/10.3390/geosciences8120438>
- Archundia, D. et al. (2017). Contaminacion de la Bahia de Cohana, Lago Titicaca (Bolivia):
  desafios y oportunidades para promover su recuperacion. *Ecologia en Bolivia*, 52(2), 65-76.
- Exploring trends and variability of water quality over Lake Titicaca using global remote
  sensing products. *Remote Sensing* (2024), 16(24), 4785.
  <https://doi.org/10.3390/rs16244785>
- Observatorio del Lago Titicaca (UMSA): <https://olt.umsa.bo/>

**Atmospheric correction over water**

- Vanhellemont, Q., Ruddick, K. (2018). Atmospheric correction of metre-scale optical
  satellite data for inland and coastal water applications. *Remote Sensing of Environment*,
  216, 586-597. (ACOLITE)
- Sentinel-2 Level-2A algorithm (SCL classes):
  <https://sentinels.copernicus.eu/web/sentinel/technical-guides/sentinel-2-msi/level-2a/algorithm>

**Tools**

- Sentinel-2 SR Harmonized: <https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED>
- Earth Engine guide - morphological operations: <https://developers.google.com/earth-engine/guides/image_morphological>
- geemap documentation: <https://geemap.org>